# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadFaizan0023/FlyRank_ML_internship_repo/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


In [2]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [ ]:
clients = con.sql(f"""
    SELECT *
    FROM {TABLES['fact_daily']}
""")


In [ ]:
print(clients)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬────────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_cl

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

I made a 90 days time window selected from latest 3 months from date 2026-04-01 to 2026-06-30.<br><br>
One client's one reported content page needed refresh or not.<br><br> Trained on 30day window train, validated on 30day window validation and tested on 30day window test, and output label: "refresh_needed" derived by comparison between features of the 30day train window and 30day validation window.

<h1>Time window</h1>

In [ ]:
date_range = con.sql(f"""
    SELECT
        MIN(report_date) as start_date,
        MAX(report_date) as end_date,
        MAX(report_date) - MIN(report_date) as duration
    FROM {TABLES['fact_daily']}
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬──────────┐
│ start_date │  end_date  │ duration │
│    date    │    date    │  int64   │
├────────────┼────────────┼──────────┤
│ 2025-01-27 │ 2026-06-30 │      519 │
└────────────┴────────────┴──────────┘



In [ ]:
con.sql(f"""
    SELECT
        count(DISTINCT report_date) as unique_days,
        min(report_date) as start,
        max(report_date) as end,
        (max(report_date) - min(report_date)) as day_diff
    FROM {TABLES['fact_daily']}
""").show()

┌─────────────┬────────────┬────────────┬──────────┐
│ unique_days │   start    │    end     │ day_diff │
│    int64    │    date    │    date    │  int64   │
├─────────────┼────────────┼────────────┼──────────┤
│         520 │ 2025-01-27 │ 2026-06-30 │      519 │
└─────────────┴────────────┴────────────┴──────────┘



### Monthly Date Density
The query below shows how many unique dates appear in each month to help identify if you have a full ~30 days of data per month.

In [ ]:
con.sql(f"""
    SELECT
        date_trunc('month', report_date) as month,
        count(DISTINCT report_date) as days_present
    FROM {TABLES['fact_daily']}
    GROUP BY 1
    ORDER BY 1
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────┐
│   month    │ days_present │
│    date    │    int64     │
├────────────┼──────────────┤
│ 2025-01-01 │            5 │
│ 2025-02-01 │           28 │
│ 2025-03-01 │           31 │
│ 2025-04-01 │           30 │
│ 2025-05-01 │           31 │
│ 2025-06-01 │           30 │
│ 2025-07-01 │           31 │
│ 2025-08-01 │           31 │
│ 2025-09-01 │           30 │
│ 2025-10-01 │           31 │
│ 2025-11-01 │           30 │
│ 2025-12-01 │           31 │
│ 2026-01-01 │           31 │
│ 2026-02-01 │           28 │
│ 2026-03-01 │           31 │
│ 2026-04-01 │           30 │
│ 2026-05-01 │           31 │
│ 2026-06-01 │           30 │
├────────────┴──────────────┤
│ 18 rows         2 columns │
└───────────────────────────┘



In [ ]:
clients_last_3m = con.sql(f"""
    SELECT report_date
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-04-01'
      AND report_date <= '2026-06-30'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
print(f"Rows in last 3 months: {len(clients_last_3m):,}")

Rows in last 3 months: 33,806,178


,report_date
0,2026-04-01
1,2026-04-01
2,2026-04-01
3,2026-04-01
4,2026-04-01


In [ ]:
# Verify the range in the new subset
con.sql("""
    SELECT
        MIN(report_date) as start_date,
        MAX(report_date) as end_date
    FROM clients_last_3m
""").show()

┌─────────────────────┬─────────────────────┐
│     start_date      │      end_date       │
│      timestamp      │      timestamp      │
├─────────────────────┼─────────────────────┤
│ 2026-04-01 00:00:00 │ 2026-06-30 00:00:00 │
└─────────────────────┴─────────────────────┘



<h1>Unit of analysis</h1>

In [4]:
unit_analysis_data = con.sql(f"""
    SELECT *
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-04-01'
      AND report_date <= '2026-06-30'
    LIMIT 1
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [6]:
print(unit_analysis_data.columns)

Index(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc',
       'client_has_ga4', 'gsc_data_available', 'ga4_data_available',
       'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position',
       'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions',
       'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct',
       'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai',
       'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude',
       'ai_meta', 'ai_other', 'scroll_events', 'month'],
      dtype='object')


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

<h1>Features:</h1>
'client_has_gsc', 'client_has_ga4', 'gsc_data_available',
       'ga4_data_available', 'gsc_impressions', 'gsc_clicks',
       'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions',
       'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec',
       'sessions_organic', 'sessions_direct', 'sessions_referral',
       'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt',
       'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta',
       'ai_other', 'scroll_events', 'content_type', 'search_volume', 'competition',
       'competition_level', 'cpc', 'main_intent', 'backlinks', 'provider_used',
       'model_used', 'char_count', 'word_count'

<h1>Label: "refresh_needed = 1/0"</h1>
Derived label from train/validation window data from the data table columns.  

<h1>Context:</h1>
client_hash_id, content_hash_id, report_date, content_created_date, content_updated_date, last_optimized_date, is_published, is_deleted,

<h1>Excluded:</h1>
1. month <br>
Reason: report_date already shows month so no need<br>
2. optimization_eligible_date<br>
Reason: leakage for output refresh_needed so dropped<br>
3. keyword_hash_id<br>
Reason: not useful for output label prediction<br>
4. url_hash_id<br>
Reason: not useful for output label prediction<br>
5. keyword_char_count<br>
Reason: not useful for output label prediction<br>
6. keyword_token_count<br>
Reason: not useful for output label prediction<br>
7. url_char_count<br>
Reason: not useful for output label prediction<br>
8. keyword_created_date<br>
Reason: not useful for output label prediction<br>
9. category_count<br>
Reason: not useful for output label prediction<br>

In [5]:
dim_c= con.sql(f"""
    SELECT *
    FROM {TABLES['dim_content']}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [8]:
dim_c.columns

Index(['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id',
       'keyword_char_count', 'keyword_token_count', 'url_char_count',
       'content_created_date', 'content_updated_date', 'content_type',
       'search_volume', 'competition', 'competition_level', 'cpc',
       'main_intent', 'backlinks', 'category_count', 'keyword_created_date',
       'provider_used', 'model_used', 'char_count', 'word_count',
       'last_optimized_date', 'optimization_eligible_date', 'is_published',
       'is_deleted'],
      dtype='object')

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
joined_data = con.sql(f"""
    SELECT
        f.*,
        d.keyword_char_count,
        d.keyword_token_count,
        d.content_created_date,
        d.content_updated_date,
        d.content_type,
        d.search_volume,
        d.competition,
        d.competition_level,
        d.cpc,
        d.main_intent,
        d.backlinks,
        d.category_count,
        d.keyword_created_date,
        d.provider_used,
        d.model_used,
        d.char_count,
        d.word_count,
        d.last_optimized_date,
        d.optimization_eligible_date,
        d.is_published,
        d.is_deleted
    FROM {TABLES['fact_daily']} f
    LEFT JOIN {TABLES['dim_content']} d
      ON f.client_hash_id = d.client_hash_id
      AND f.content_hash_id = d.content_hash_id
    WHERE f.report_date >= '2026-04-01'
      AND f.report_date <= '2026-04-30'
    LIMIT 1000000
""").df()

display(joined_data)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [44]:
joined_data.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc',
       'client_has_ga4', 'gsc_data_available', 'ga4_data_available',
       'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position',
       'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions',
       'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct',
       'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai',
       'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude',
       'ai_meta', 'ai_other', 'scroll_events', 'month', 'keyword_char_count',
       'keyword_token_count', 'content_created_date', 'content_updated_date',
       'content_type', 'search_volume', 'competition', 'competition_level',
       'cpc', 'main_intent', 'backlinks', 'category_count',
       'keyword_created_date', 'provider_used', 'model_used', 'char_count',
       'word_count', 'last_optimized_date', 'optimization_eligible_date',
       'is_published', '

In [45]:
joined_data = joined_data.drop(columns=['report_date', 'client_hash_id', 'content_hash_id','month','keyword_char_count',
                                        'keyword_token_count', 'category_count','keyword_created_date','last_optimized_date', 'optimization_eligible_date' ])

In [47]:
joined_data.columns

Index(['client_has_gsc', 'client_has_ga4', 'gsc_data_available',
       'ga4_data_available', 'gsc_impressions', 'gsc_clicks',
       'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions',
       'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec',
       'sessions_organic', 'sessions_direct', 'sessions_referral',
       'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt',
       'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta',
       'ai_other', 'scroll_events', 'content_created_date',
       'content_updated_date', 'content_type', 'search_volume', 'competition',
       'competition_level', 'cpc', 'main_intent', 'backlinks', 'provider_used',
       'model_used', 'char_count', 'word_count', 'is_published', 'is_deleted'],
      dtype='object')

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 3.1 Grain Verification
Check if the grain is truly (date, client, content) by looking for duplicates.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

con.sql(f"""
    SELECT
        report_date, client_hash_id, content_hash_id, COUNT(*)
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-04-01' AND report_date <= '2026-04-30'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
""").show()


### 3.2 Missing Values Check


In [52]:
# Check null counts for key features in our defined window for 1Million instances
display(joined_data.isnull().sum())

,0
client_has_gsc,0
client_has_ga4,0
gsc_data_available,0
ga4_data_available,162250
gsc_impressions,0
gsc_clicks,0
gsc_sum_position,0
gsc_avg_position,753105
ga4_pageviews,162250
ga4_sessions,162250


### 3.3 Window & Count
Confirm the specific boundaries and row counts for the joined set.
Here: count is taken of 1Million instances not full window.<br>
Full 90day window instances: 33,806,178

In [51]:
print(f"Total Rows: {len(joined_data):,}")
# We check the original fact table for the window since joined_data had IDs dropped
con.sql(f"""
    SELECT
        MIN(report_date) as start,
        MAX(report_date) as end
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-04-01' AND report_date <= '2026-06-30'
""").show()

Total Rows: 1,000,000


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┐
│   start    │    end     │
│    date    │    date    │
├────────────┼────────────┤
│ 2026-04-01 │ 2026-06-30 │
└────────────┴────────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### 4. Data Limits Summary
1. **Metric Sparsity:** A significant portion of the dataset contains GSC impressions but lacks GA4 engagement metrics (pageviews/sessions), meaning the model cannot reliably predict engagement-based needs for all pages (only those with GA4 connected).
2. **Temporal Resolution:** The model uses daily snapshots, but seasonal trends, holidays, or sudden Google algorithm updates aren't explicitly encoded as features, making it 'blind' to macro-economic changes.
3. **Snapshot Latency:** Since `dim_content` provides the current state of a page (e.g., current word count), the model cannot see historical changes to the content itself, only the performance impact resulting from those changes.

In [53]:
# Quantifying the GSC vs GA4 availability limit
con.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(gsc_impressions) as rows_with_gsc,
        COUNT(ga4_pageviews) as rows_with_ga4,
        (COUNT(ga4_pageviews) * 100.0 / COUNT(*)) as ga4_coverage_pct
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-04-01' AND report_date <= '2026-04-30'
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────┬───────────────┬───────────────────┐
│ total_rows │ rows_with_gsc │ rows_with_ga4 │ ga4_coverage_pct  │
│   int64    │     int64     │     int64     │      double       │
├────────────┼───────────────┼───────────────┼───────────────────┤
│   10424730 │      10424730 │       8208679 │ 78.74236550970625 │
└────────────┴───────────────┴───────────────┴───────────────────┘



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.